# Notebook 04 — MySQL Integration


## 1. Import libraries and configure paths

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import getpass
import json
import os
import re

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.exc import SQLAlchemyError

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas version:", pd.__version__)


Pandas version: 3.1.0


In [2]:
cwd = Path.cwd().resolve()

if (cwd / "data" / "processed").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").is_dir() and cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = next(
        (p for p in [cwd, *cwd.parents] if (p / "data" / "processed").is_dir()),
        None
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed. Run Notebook 02 first and open this notebook "
        "from the project root or notebooks/ folder."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
processed_candidates = [
    p for p in PROCESSED_DIR.glob("usgs_earthquakes_cleaned_*.csv")
    if p.is_file()
]
if not processed_candidates:
    raise FileNotFoundError(
        f"No cleaned CSV found in {PROCESSED_DIR}. Run Notebook 02 successfully first."
    )

processed_file = max(processed_candidates, key=lambda p: p.stat().st_mtime)
print("Project root:", PROJECT_ROOT)
print("Input CSV:", processed_file.name)
print("File size (MB):", round(processed_file.stat().st_size / (1024 ** 2), 2))


Project root: D:\global seismic trends
Input CSV: usgs_earthquakes_cleaned_20261022T134538Z.csv
File size (MB): 373.63


## 2. Configure the MySQL connection

In [3]:
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306

MYSQL_USER = "root"
MYSQL_PASSWORD = "12345"

DATABASE_NAME = "global_seismic_trends"
TABLE_NAME = "earthquakes"

server_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    query={"charset": "utf8mb4"}
)

try:
    server_engine = create_engine(
        server_url,
        pool_pre_ping=True,
        connect_args={"connect_timeout": 10}
    )

    with server_engine.connect() as conn:
        version = conn.execute(text("SELECT VERSION()")).scalar()
        print("Connected to MySQL version:", version)

except SQLAlchemyError as exc:
    raise RuntimeError(
        "Could not connect to MySQL. Check whether MySQL80 is running "
        "and verify your credentials. Original error: " + str(exc)
    ) from exc

Connected to MySQL version: 8.0.45


## 3. Create the project database

In [4]:
with server_engine.begin() as conn:
    conn.execute(text(
        f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}` "
        "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
    ))

print(f"Database '{DATABASE_NAME}' is available.")


Database 'global_seismic_trends' is available.


## 4. Load and prepare the cleaned CSV

In [ ]:
df = pd.read_csv(processed_file, low_memory=False)

for col in ["time", "updated"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], utc=True, errors="coerce")
        df[col] = parsed.dt.tz_convert("UTC").dt.tz_localize(None)

for col in df.columns:
    if str(df[col].dtype) in ("Int64", "Int32", "UInt64", "UInt32"):
        df[col] = df[col].astype("object").where(df[col].notna(), None)
    elif str(df[col].dtype) == "boolean":
        df[col] = df[col].astype("object").where(df[col].notna(), None)

# Ensure column names are safe MySQL identifiers.
def mysql_identifier(name):
    cleaned = re.sub(r"[^0-9a-zA-Z_]", "_", str(name))
    if not cleaned:
        cleaned = "column"
    if cleaned[0].isdigit():
        cleaned = "_" + cleaned
    return cleaned[:64]

renamed_columns = {col: mysql_identifier(col) for col in df.columns}
if len(set(renamed_columns.values())) != len(renamed_columns):
    raise ValueError("Column-name sanitization produced duplicate names; inspect the input schema.")
df = df.rename(columns=renamed_columns)

print("Rows loaded from CSV:", f"{len(df):,}")
print("Columns prepared:", len(df.columns))
display(df.head())
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)}))


Rows loaded from CSV: 739,554
Columns prepared: 51


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type,alert,felt,cdi,mmi,url,detail,place_type,time_zone,is_invalid_latitude,is_invalid_longitude,has_invalid_coordinates,is_negative_depth,is_extreme_depth_over_700km,is_magnitude_outside_review_range,has_missing_magnitude,has_missing_timestamp,year,month,day,day_of_week,hour_utc,date_utc,depth_category,magnitude_category,place_region_text
0,nc73642735,2021-10-22 13:22:28.170,2021-10-22 15:26:11.754,38.807835,-122.813835,0.08,0.84,md,"6km NW of The Geysers, CA",automatic,0,11,nc,11.0,0.01203,0.02,77.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",nc73642735,",",nc,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/nc73642735,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc73642735&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"NW of The Geysers, CA"
1,mb80527769,2021-10-22 13:27:15.120,2021-10-22 15:21:51.820,47.839167,-114.289000,3.17,-0.20,ml,"3 km SSW of Dayton, Montana",reviewed,0,0,mb,6.0,0.09300,0.04,169.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527769,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527769,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527769&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"SSW of Dayton, Montana"
2,mb80527774,2021-10-22 13:31:59.150,2021-10-22 15:28:57.720,47.766833,-113.753667,3.91,0.56,ml,"19 km SSE of Swan Lake, Montana",reviewed,0,5,mb,7.0,0.19900,0.11,148.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527774,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527774,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527774&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"SSE of Swan Lake, Montana"
3,ak021dk3jgyc,2021-10-22 13:44:11.726,2021-12-06 18:54:05.882,59.678800,-136.572300,24.00,1.20,ml,"31 km WNW of Mosquito Lake, Alaska",reviewed,0,22,ak,NaN,NaN,0.26,NaN,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",ak021dk3jgyc,",",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ak021dk3jgyc,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ak021dk3jgyc&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"WNW of Mosquito Lake, Alaska"
4,ci39840223,2021-10-22 13:55:21.010,2021-10-22 17:59:28.696,33.344000,-116.928500,18.04,0.79,ml,"6km WSW of Palomar Observatory, CA",reviewed,0,10,ci,25.0,0.05595,0.15,48.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",ci39840223,",",ci,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ci39840223,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci39840223&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"WSW of Palomar Observatory, CA"


,column,dtype
id,id,str
time,time,datetime64[us]
updated,updated,datetime64[us]
latitude,latitude,float64
longitude,longitude,float64
depth_km,depth_km,float64
mag,mag,float64
magType,magType,str
place,place,str
status,status,str


## 5. Connect to the project database and inspect existing table

In [6]:
database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=DATABASE_NAME,
    query={"charset": "utf8mb4"}
)
engine = create_engine(database_url, pool_pre_ping=True, connect_args={"connect_timeout": 10})

with engine.connect() as conn:
    existing_tables = conn.execute(text("SHOW TABLES")).fetchall()

print("Existing tables:", [row[0] for row in existing_tables] if existing_tables else "None")


Existing tables: None


## 6. Import the dataset into MySQL

In [7]:
expected_rows = len(df)
CHUNK_SIZE = 10_000

try:
    df.to_sql(
        name=TABLE_NAME,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=CHUNK_SIZE,
        method="multi"
    )
    print(f"Imported {expected_rows:,} rows into {DATABASE_NAME}.{TABLE_NAME}.")
except Exception as exc:
    raise RuntimeError(
        "MySQL import failed. Check the error message, MySQL permissions, available disk space, "
        "and whether any text column exceeds MySQL's inferred column size. "
        "The table may have been partially created; rerun after correcting the issue. "
        f"Original error: {exc}"
    ) from exc


Imported 739,554 rows into global_seismic_trends.earthquakes.


## 7. Validate the imported table

In [8]:
with engine.connect() as conn:
    actual_rows = conn.execute(text(f"SELECT COUNT(*) FROM `{TABLE_NAME}`")).scalar()
    table_columns = conn.execute(text(f"SHOW COLUMNS FROM `{TABLE_NAME}`")).fetchall()
    distinct_ids = conn.execute(text(
        f"SELECT COUNT(DISTINCT `id`) FROM `{TABLE_NAME}`"
    )).scalar() if "id" in df.columns else None
    null_ids = conn.execute(text(
        f"SELECT COUNT(*) FROM `{TABLE_NAME}` WHERE `id` IS NULL"
    )).scalar() if "id" in df.columns else None
    min_time = conn.execute(text(
        f"SELECT MIN(`time`) FROM `{TABLE_NAME}`"
    )).scalar() if "time" in df.columns else None
    max_time = conn.execute(text(
        f"SELECT MAX(`time`) FROM `{TABLE_NAME}`"
    )).scalar() if "time" in df.columns else None

validation = {
    "expected_csv_rows": int(expected_rows),
    "mysql_rows": int(actual_rows),
    "row_count_matches": bool(actual_rows == expected_rows),
    "distinct_event_ids": int(distinct_ids) if distinct_ids is not None else None,
    "null_event_ids": int(null_ids) if null_ids is not None else None,
    "minimum_event_time_utc": str(min_time) if min_time is not None else None,
    "maximum_event_time_utc": str(max_time) if max_time is not None else None,
    "mysql_column_count": len(table_columns)
}
display(pd.Series(validation, name="value").to_frame())
display(pd.DataFrame(table_columns, columns=["Field", "Type", "Null", "Key", "Default", "Extra"]))

if actual_rows != expected_rows:
    raise AssertionError(
        f"Row-count mismatch: CSV has {expected_rows:,}, MySQL has {actual_rows:,}."
    )
print("Row-count validation passed.")


,value
expected_csv_rows,739554
mysql_rows,739554
row_count_matches,True
distinct_event_ids,739554
null_event_ids,0
minimum_event_time_utc,2021-10-22 13:22:28
maximum_event_time_utc,2026-10-22 13:13:42
mysql_column_count,51


,Field,Type,Null,Key,Default,Extra
0,id,text,YES,,None,
1,time,datetime,YES,,None,
2,updated,datetime,YES,,None,
3,latitude,double,YES,,None,
4,longitude,double,YES,,None,
5,depth_km,double,YES,,None,
6,mag,double,YES,,None,
7,magType,text,YES,,None,
8,place,text,YES,,None,
9,status,text,YES,,None,


Row-count validation passed.


## 8. Preview records from MySQL

In [9]:
with engine.connect() as conn:
    sample_from_mysql = pd.read_sql(
        text(
            f"SELECT `id`, `time`, `mag`, `depth_km`, `place`, `tsunami` "
            f"FROM `{TABLE_NAME}` LIMIT 10"
        ),
        conn
    )

display(sample_from_mysql)


,id,time,mag,depth_km,place,tsunami
0,nc73642735,2021-10-22 13:22:28,0.84,0.08,"6km NW of The Geysers, CA",0
1,mb80527769,2021-10-22 13:27:15,-0.20,3.17,"3 km SSW of Dayton, Montana",0
2,mb80527774,2021-10-22 13:31:59,0.56,3.91,"19 km SSE of Swan Lake, Montana",0
3,ak021dk3jgyc,2021-10-22 13:44:12,1.20,24.00,"31 km WNW of Mosquito Lake, Alaska",0
4,ci39840223,2021-10-22 13:55:21,0.79,18.04,"6km WSW of Palomar Observatory, CA",0
5,nc73642745,2021-10-22 13:55:28,2.14,2.70,"7km ENE of Walker, CA",0
6,nc73642755,2021-10-22 13:58:01,0.82,10.27,"11km NE of San Martin, CA",0
7,nc73642750,2021-10-22 13:59:38,1.17,3.37,"7km E of Walker, CA",0
8,nc73642760,2021-10-22 14:02:34,1.10,3.71,"7km E of Walker, CA",0
9,us6000g9mz,2021-10-22 14:06:00,3.00,16.00,"110 km SSE of Sand Point, Alaska",0


## 9. Save import metadata

In [10]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
metadata_path = PROCESSED_DIR / f"mysql_import_metadata_{run_timestamp}.json"

import_metadata = {
    "project": "Global Seismic Trends: Data-Driven Earthquake Insights",
    "notebook": "04_mysql_integration.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_csv": str(processed_file),
    "database": DATABASE_NAME,
    "table": TABLE_NAME,
    "mysql_host": MYSQL_HOST,
    "mysql_port": MYSQL_PORT,
    "expected_csv_rows": int(expected_rows),
    "mysql_rows": int(actual_rows),
    "row_count_matches": bool(actual_rows == expected_rows),
    "column_count": int(len(table_columns)),
    "credentials_saved": False
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(import_metadata, f, indent=2, ensure_ascii=False)

print("Import metadata saved to:", metadata_path)


Import metadata saved to: D:\global seismic trends\data\processed\mysql_import_metadata_20261023T103143Z.json


## 10. Clean up connections

In [11]:
engine.dispose()
server_engine.dispose()
print("SQLAlchemy connections disposed.")


SQLAlchemy connections disposed.
